In [ ]:
import gzip
import os
import requests
import tarfile
from tqdm import tqdm

from src.utils import ensure_dir, sha256_file

In [ ]:
CONFIG = {
    "uniprot_release": "2025_03",
    "train_terms_path": "comp_data/Train/train_terms.tsv",
    "train_output_filename": "train_sequences.fasta",
    "chunk_size": 1024 * 1024 * 16,  # 16 MB is a good default for large downloads
    "output_base_dir": "./input_data",
}

In [ ]:
def parse_uniprot_header(line):
    """
    Parse a UniProt FASTA header line like:
    >sp|P12345|ENTRY_NAME ...
    Returns the accession ID, or None if it cannot be parsed.
    """
    parts = line.strip().split("|")
    if len(parts) > 1:
        return parts[1]
    return None


def fasta_open(path, mode="rt"):
    """
    Open .gz or plain text FASTA transparently.
    """
    if path.endswith(".gz"):
        return gzip.open(path, mode)
    return open(path, mode)


def download_uniprot_sprot():
    """
    Download the release-specific Swiss-Prot archive from UniProt previous releases.
    For archived releases, UniProt provides tarballs rather than the same current-release
    convenience layout. :contentReference[oaicite:2]{index=2}
    """
    release = CONFIG["uniprot_release"]
    base_url = "https://ftp.uniprot.org/pub/databases/uniprot"
    release_str = f"previous_releases/release-{release}/knowledgebase"
    tar_filename = f"uniprot_sprot-only{release}.tar.gz"
    url = f"{base_url}/{release_str}/{tar_filename}"

    output_dir = os.path.join(CONFIG["output_base_dir"], "uniprot", release)
    ensure_dir(output_dir)

    tar_output_path = os.path.join(output_dir, tar_filename)
    extracted_fasta_gz_path = os.path.join(output_dir, "uniprot_sprot.fasta.gz")

    # Fast path: extracted file already exists
    if os.path.exists(extracted_fasta_gz_path):
        print(f"[INFO] Extracted FASTA already exists: {extracted_fasta_gz_path}")
        return extracted_fasta_gz_path

    # Download TAR if needed
    if not os.path.exists(tar_output_path):
        print(f"[INFO] Downloading UniProt Swiss-Prot ({release})...")

        response = requests.get(url, stream=True)
        response.raise_for_status()

        total_size = int(response.headers.get("content-length", 0))

        with open(tar_output_path, "wb") as f, tqdm(
            total=total_size,
            unit="B",
            unit_scale=True
        ) as pbar:
            for chunk in response.iter_content(CONFIG["chunk_size"]):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

        print(f"[INFO] TAR file download completed: {tar_output_path}")
    else:
        print(f"[INFO] TAR already exists: {tar_output_path}")

    # Extract only if needed
    print("[INFO] Extracting FASTA from TAR...")
    with tarfile.open(tar_output_path, "r:gz") as tar:
        for member in tar.getmembers():
            if member.name.endswith(".fasta.gz"):
                member.name = os.path.basename(member.name)
                tar.extract(member, output_dir)
                print(f"[INFO] Extracted: {member.name}")
                return os.path.join(output_dir, member.name)

    raise FileNotFoundError("FASTA file not found in the tar package.")


def load_train_protein_ids():
    """
    Stream train_terms.tsv and collect unique protein IDs from the first column.
    This avoids loading the full TSV into a DataFrame when only the accession set is needed.
    """
    print("[INFO] Loading train_terms.tsv...")

    protein_ids = set()

    with open(CONFIG["train_terms_path"], "r", encoding="utf-8") as f:
        header = next(f, None)  # skip header
        if header is None:
            raise ValueError("train_terms.tsv is empty.")

        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            protein_id = line.split("\t", 1)[0]
            protein_ids.add(protein_id)

    print(f"[INFO] Unique proteins in train_terms: {len(protein_ids)}")
    return protein_ids


def filter_fasta_by_ids(fasta_path, protein_ids):
    """
    Stream the source FASTA and write only records whose accession is in protein_ids.
    Returns:
        output_path, found_ids
    """
    release = CONFIG["uniprot_release"]
    output_dir = os.path.join(CONFIG["output_base_dir"], "train", release)
    ensure_dir(output_dir)

    output_path = os.path.join(output_dir, CONFIG["train_output_filename"])

    print("[INFO] Filtering FASTA...")

    kept = 0
    total = 0
    found_ids = set()

    with fasta_open(fasta_path, "rt") as infile, open(output_path, "w", encoding="utf-8") as outfile:
        write_flag = False

        for line in infile:
            if line.startswith(">"):
                total += 1
                protein_id = parse_uniprot_header(line)

                if protein_id in protein_ids:
                    write_flag = True
                    kept += 1
                    found_ids.add(protein_id)
                    outfile.write(line)
                else:
                    write_flag = False
            else:
                if write_flag:
                    outfile.write(line)

    print(f"[INFO] Total sequences scanned: {total}")
    print(f"[INFO] Sequences kept: {kept}")

    return output_path, found_ids


def validate_train_fasta(found_ids, expected_ids):
    """
    Validate using the IDs collected during filtering, so we avoid rereading the output FASTA.
    """
    print("\n[INFO] Validating filtered FASTA...")

    print(f"[INFO] Found sequences: {len(found_ids)}")

    missing = expected_ids - found_ids
    extra = found_ids - expected_ids

    print(f"[INFO] Missing sequences: {len(missing)}")
    print(f"[INFO] Extra sequences: {len(extra)}")

    if not missing and not extra:
        print("[SUCCESS] FASTA matches expected protein IDs")
        return True

    print("[FAIL] Mismatch detected")

    if missing:
        preview_missing = sorted(list(missing))[:10]
        print(f"[INFO] Sample missing IDs: {preview_missing}")

    if extra:
        preview_extra = sorted(list(extra))[:10]
        print(f"[INFO] Sample extra IDs: {preview_extra}")

    return False


def load_fasta_id_to_sequence(path):
    """
    Load a FASTA file into a mapping: accession -> full sequence.
    Works for plain text and .gz FASTA.
    Used only in the semantic fallback comparison.
    """
    seqs = {}
    current_id = None
    current_seq = []

    with fasta_open(path, "rt") as f:
        for line in f:
            if line.startswith(">"):
                if current_id is not None:
                    seqs[current_id] = "".join(current_seq)

                current_id = parse_uniprot_header(line)
                current_seq = []
            else:
                current_seq.append(line.strip())

        if current_id is not None:
            seqs[current_id] = "".join(current_seq)

    return seqs


def compare_fasta_files(path1, path2):
    """
    Compare:
      1) byte hash
      2) accession set
      3) accession -> sequence mapping
    """
    print("\n[INFO] Comparing FASTA files...")

    hash1 = sha256_file(path1)
    hash2 = sha256_file(path2)

    print(f"[INFO] Hash1: {hash1}")
    print(f"[INFO] Hash2: {hash2}")

    if hash1 == hash2:
        print("[SUCCESS] Files are IDENTICAL")
        return True

    print("[WARN] Binary mismatch → checking ID and sequence content...")

    seqs1 = load_fasta_id_to_sequence(path1)
    seqs2 = load_fasta_id_to_sequence(path2)

    ids1 = set(seqs1.keys())
    ids2 = set(seqs2.keys())

    print(f"[INFO] IDs1: {len(ids1)}")
    print(f"[INFO] IDs2: {len(ids2)}")

    missing = ids1 - ids2
    extra = ids2 - ids1

    print(f"[INFO] Missing IDs: {len(missing)}")
    print(f"[INFO] Extra IDs: {len(extra)}")

    if missing:
        print(f"[INFO] Sample missing IDs: {sorted(list(missing))[:10]}")
    if extra:
        print(f"[INFO] Sample extra IDs: {sorted(list(extra))[:10]}")

    if ids1 != ids2:
        print("[FAIL] Sequence ID mismatch")
        return False

    sequence_mismatches = []
    for protein_id in ids1:
        if seqs1[protein_id] != seqs2[protein_id]:
            sequence_mismatches.append(protein_id)
            if len(sequence_mismatches) >= 10:
                break

    print(f"[INFO] Sequence mismatches: {len(sequence_mismatches)}")
    if sequence_mismatches:
        print(f"[INFO] Sample sequence mismatch IDs: {sequence_mismatches}")
        print("[FAIL] Same IDs but different sequence content")
        return False

    print("[SUCCESS] Same protein IDs and same sequence content")
    return True

In [ ]:
sprot_path = download_uniprot_sprot()
protein_ids = load_train_protein_ids()

In [ ]:
filtered_fasta, found_ids = filter_fasta_by_ids(sprot_path, protein_ids)

In [ ]:
validate_train_fasta(found_ids, protein_ids)

In [ ]:
reference_path = "comp_data/Train/train_sequences.fasta"
compare_fasta_files(reference_path, filtered_fasta)